In [1]:
import os
from datetime import datetime
import shutil
from pathlib import Path
import pandas as pd
import datasets
from datasets import DatasetDict, Dataset
from multitask_trainer import train_eval_model

In [2]:
cache_dir = '../cache'
data_dir = Path('data/hf_berkeley_multi')
output_dir = Path('data/berkeley_dataset_multi_augmented')
TOXIGEN_MAP = {
    'sexuality': ['lgbtq'],
    'disability': ['mental_dis', 'physical_dis']
}

In [3]:
data = datasets.load_dataset("toxigen/toxigen-data", cache_dir=cache_dir)
train_df = data['train'].to_pandas()
shutil.copytree(data_dir, output_dir, dirs_exist_ok=True)
for group_name, tox_targets in TOXIGEN_MAP.items():
    task_dir = f'target={group_name}'
    dataset = DatasetDict.load_from_disk(str(data_dir/task_dir))
    df_aug = train_df[train_df['target_group'].isin(tox_targets)]
    df_aug = pd.DataFrame({'text': df_aug['text'], 'value': 1, 'task_name': group_name})
    df_aug = Dataset.from_pandas(df_aug).cast_column('value', dataset['train'].features['value'])
    dataset['train'] = datasets.concatenate_datasets([dataset['train'], df_aug])
    dataset.save_to_disk(str(output_dir/task_dir))

Casting the dataset:   0%|          | 0/714 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/28409 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5935 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5935 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1399 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/29094 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5935 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5935 [00:00<?, ? examples/s]

## LoRa

In [4]:
os.environ["CUDA_VISIBLE_DEVICES"]="1"
model_path = "intfloat/e5-large-v2"
model_dir = "outputs/berkeley_toxigen_classifier_multi_e5_large"

In [5]:
print("Training started:", datetime.now())

Training started: 2026-01-18 18:22:38.932753


In [6]:
lora_args = {
    "init_lora_weights": "olora",
    "target_modules": "all-linear"
}
_, results = train_eval_model(model_path, data_dir=output_dir, cache_dir=cache_dir, 
                              save_model_dir=model_dir, output_dir="test-classifier-lora-large", 
                              batch_size=64, num_epochs=20, eval_batch_size=128, 
                              use_lora=True, lora_args=lora_args)
print(results)

Some weights of BertMultiTaskClassifier were not initialized from the model checkpoint at intfloat/e5-large-v2 and are newly initialized: ['heads.0.1.bias', 'heads.0.1.weight', 'heads.1.1.bias', 'heads.1.1.weight', 'heads.2.1.bias', 'heads.2.1.weight', 'heads.3.1.bias', 'heads.3.1.weight', 'heads.4.1.bias', 'heads.4.1.weight', 'heads.5.1.bias', 'heads.5.1.weight', 'heads.6.1.bias', 'heads.6.1.weight', 'heads.7.1.bias', 'heads.7.1.weight', 'heads.8.1.bias', 'heads.8.1.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The model is already on multiple devices. Skipping the move to device specified in `args`.
2026/01/18 18:22:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/18 18:22:46 INFO mlflow.store.db.utils: Updating database tables
2026/01/18 18:22:46 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/18 18:22:46 INFO alembic.runtime.migration: Will assume non-transact

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Precision,Recall,Roc Auc
1,2.043700,0.197886,0.927829,0.732634,0.836666,0.730075,0.783382
2,1.734600,0.199103,0.927829,0.778595,0.765081,0.797685,0.817407
3,1.644200,0.192898,0.930619,0.772709,0.803389,0.769262,0.805363
4,1.583900,0.194429,0.929477,0.779491,0.806881,0.773101,0.807149
5,1.522000,0.193317,0.932191,0.789403,0.821836,0.772853,0.809081
6,1.457300,0.195234,0.931611,0.795078,0.804639,0.794644,0.819342
7,1.410400,0.199674,0.929533,0.792196,0.803507,0.790062,0.813811
8,1.362100,0.200256,0.930394,0.794304,0.805410,0.800278,0.820980
9,1.305200,0.202468,0.930619,0.798724,0.810101,0.794223,0.819766
10,1.250500,0.205614,0.931143,0.789172,0.811062,0.784684,0.815705



***** Running Evaluation *****
  Num examples = 53415
  Batch size = 128
Saving model checkpoint to test-classifier-lora-large/checkpoint-3928
loading configuration file config.json from cache at /home/paudan/.cache/huggingface/hub/models--intfloat--e5-large-v2/snapshots/f169b11e22de13617baa190a028a32f3493550b6/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.5",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

Deleting older checkpoint [test-classifier-lora-large/checkpoint-42845]

hatespeech
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      4396
           1       0.00      0.00      0.00       290
           2       0.69      0.56      0.61      1249

    accuracy                           0.81      5935
   macro avg       0.51      0.50      0.50      5935
weighted avg       0.77      0.81      0.79      5935

gender
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      4010
           1       0.86      0.86      0.86      1925

    accuracy                           0.91      5935
   macro avg       0.89      0.89      0.89      5935
weighted avg       0.91      0.91      0.91      5935

sexuality
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      4972
           1       0.88      0.77      0.82       963

    accuracy                           0.95      5935
   macro avg       0.92      0.87      0.89   

/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


age
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      5834
           1       0.65      0.30      0.41       101

    accuracy                           0.99      5935
   macro avg       0.82      0.65      0.70      5935
weighted avg       0.98      0.99      0.98      5935

religion
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      5082
           1       0.86      0.86      0.86       853

    accuracy                           0.96      5935
   macro avg       0.92      0.92      0.92      5935
weighted avg       0.96      0.96      0.96      5935

contains_hate
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       297
           1       0.95      1.00      0.97      5638

    accuracy                           0.95      5935
   macro avg       0.47      0.50      0.49      5935
weighted avg       0.90      0.95      0.93    

tokenizer config file saved in outputs/berkeley_toxigen_classifier_multi_e5_large/tokenizer_config.json
Special tokens file saved in outputs/berkeley_toxigen_classifier_multi_e5_large/special_tokens_map.json
loading configuration file config.json from cache at /home/paudan/.cache/huggingface/hub/models--intfloat--e5-large-v2/snapshots/f169b11e22de13617baa190a028a32f3493550b6/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.5",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



            task  accuracy  f1_score  precision    recall     kappa   roc_auc
0     hatespeech  0.814490  0.814490   0.814490  0.814490  0.469959       NaN
1         gender  0.907498  0.857440   0.857217  0.857662  0.788977  0.894542
2      sexuality  0.945240  0.820343   0.877069  0.770509  0.788199  0.874796
3            age  0.985341  0.408163   0.652174  0.297030  0.401792  0.647144
4       religion  0.960067  0.860997   0.861502  0.860492  0.837682  0.918637
5  contains_hate  0.949789  0.974248   0.949949  0.999823 -0.000336  0.499911
6         origin  0.924347  0.769389   0.723671  0.821272  0.724356  0.882167
7           race  0.910025  0.810100   0.901821  0.735313  0.751943  0.853521
8     disability  0.987195  0.709924   0.673913  0.750000  0.703396  0.871128


In [7]:
print("Training finished:", datetime.now())

Training finished: 2026-01-19 19:37:59.461138


## Evaluation on HateCheck

In [15]:
from multitask_trainer import evaluate_model
# from multitask_trainer import calculate_scores
evaluate_model(model_dir, 'data/hatecheck_dataset_multi', cache_dir)

loading file vocab.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
loading configuration file config.json from cache at ../cache/models--intfloat--e5-large-v2/snapshots/f169b11e22de13617baa190a028a32f3493550b6/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.5",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at ..

Task: gender
Task: sexuality
Task: religion
Task: contains_hate
Task: origin
Task: race
Task: disability


/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/conda/lib/python3.12/site

,task,accuracy,f1_score,precision,recall,kappa,roc_auc
0,gender,0.968566,0.968566,0.968566,0.968566,0.0,NaN
1,sexuality,0.769231,0.769231,0.769231,0.769231,0.0,NaN
2,religion,0.900826,0.900826,0.900826,0.900826,0.0,NaN
3,contains_hate,0.687500,0.814815,0.687500,1.000000,0.0,0.5
4,origin,0.896328,0.896328,0.896328,0.896328,0.0,NaN
5,race,0.946058,0.946058,0.946058,0.946058,0.0,NaN
6,disability,0.927686,0.927686,0.927686,0.927686,0.0,NaN
